# Perseptron v2 - 02 Image-History MLP

Bu notebook gorsel baseline'i egitir. Model tabular metadata kullanmaz; aday urun EfficientNet-B0 embedding'i, musterinin gecmis gorsel profil embedding'i, cosine similarity ve history count sinyallerini kullanir.

Bu model proposal'daki image-only fikrine kisisellestirilmis recommender karsiligidir.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_source_project():
    candidates = [Path('/kaggle/working/perseptron_project'), Path.cwd(), Path('/kaggle/input/perseptron-project')]
    candidates += [path for path in Path('/kaggle/input').glob('*') if path.is_dir()]
    for path in candidates:
        if (path / 'src' / 'proposal_v2').exists():
            return path
    return Path.cwd()

SOURCE_PROJECT_DIR = find_source_project()
WORK_PROJECT_DIR = Path('/kaggle/working/perseptron_project_work') if Path('/kaggle').exists() else SOURCE_PROJECT_DIR
if SOURCE_PROJECT_DIR != WORK_PROJECT_DIR:
    shutil.copytree(SOURCE_PROJECT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

PROJECT_DIR = WORK_PROJECT_DIR
os.environ['PERSEPTRON_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('SOURCE_PROJECT_DIR =', SOURCE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

def restore_previous_outputs():
    if not Path('/kaggle/input').exists():
        return
    for input_root in Path('/kaggle/input').glob('*'):
        if input_root == SOURCE_PROJECT_DIR:
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = input_root / relative
            target = PROJECT_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print('Restored', source, '->', target)

restore_previous_outputs()

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


## Parametreler

`FAST_RUN=False` final fold kosulari icindir. Embedding cache Kaggle inputlarinda bulunmalidir.

In [ ]:
FAST_RUN = True
FOLD_ID = 0
MAX_TRAIN_POSITIVES = 5_000 if FAST_RUN else 200_000
MAX_VAL_POSITIVES = 1_000 if FAST_RUN else 50_000
EPOCHS = 1 if FAST_RUN else 3
BATCH_SIZE = 512 if FAST_RUN else 4096


## Egitim

Bu hucre sadece `image_history` modelini egitir. Modelin gucu, musterinin onceki satin aldigi urunlerin nasil gorundugunu temsil edebilmesinden gelir.

In [ ]:
run_module(
    'src.proposal_v2.train',
    '--fold-id', FOLD_ID,
    '--models', 'image_history',
    '--max-train-positives', MAX_TRAIN_POSITIVES,
    '--max-val-positives', MAX_VAL_POSITIVES,
    '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE,
)
